# 03 - Favorita demand exploration

Kaggle Store Sales, the training data for **Model B** (multi-quantile demand).

This notebook runs against the real download when `data/raw/favorita` is present,
and otherwise falls back to the synthetic fixture used by the test suite, so it
always executes top-to-bottom. The banner below says which one you are looking at.


In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from twinmind.utils.config import load_config
from twinmind.utils.paths import figures_dir
from twinmind.utils.seed import set_seed

set_seed(42)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
FIGURES = figures_dir()
DPI = 150

def save(fig, name):
    """Save a figure to reports/figures as a 150-dpi PNG."""
    path = FIGURES / f"{name}.png"
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    print(f"saved {path.relative_to(path.parents[2])}")
    return path

config = load_config("data")


In [ ]:
from twinmind.data.features import (
    build_demand_features, make_rolling_origin_splits, prepare_oil,
)
from twinmind.data.loaders import DataNotDownloadedError, load_favorita

cfg = config.favorita

try:
    frames = load_favorita()
    SOURCE = "real"
except DataNotDownloadedError as exc:
    sys.path.insert(0, str(FIGURES.parents[1] / "tests"))
    from synthetic import make_favorita
    frames = make_favorita(n_stores=8, days=900)
    SOURCE = "synthetic"
    print(f"NOTE: {exc}\n")

print(f"data source: {SOURCE.upper()}")
for name, frame in frames.items():
    print(f"  {name:<14} {frame.shape}")

In [ ]:
features = build_demand_features(frames, cfg, top_n=cfg.top_series)
print(f"{len(features):,} rows | {features['unique_id'].nunique()} series "
      f"| {features.shape[1]} columns")
print(f"{features['ds'].min().date()} to {features['ds'].max().date()}")
features.head()

## Total sales over time

In [ ]:
daily = features.groupby("ds")["y"].sum()
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(daily.index, daily.to_numpy(), lw=0.8, color="#4C72B0")
ax.plot(daily.index, daily.rolling(28, min_periods=1).mean().to_numpy(),
        lw=2, color="#C44E52", label="28-day mean")
ax.set_title(f"Total daily sales across {features['unique_id'].nunique()} series "
             f"({SOURCE} data)", fontweight="bold")
ax.set_xlabel("date"); ax.set_ylabel("units"); ax.legend()
fig.tight_layout()
save(fig, "03_favorita_total_sales");

## Weekly and yearly seasonality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
by_dow = features.groupby("dow")["y"].mean()
axes[0].bar(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"], by_dow.to_numpy(),
            color="#4C72B0", edgecolor="white")
axes[0].set_title("Mean sales by day of week"); axes[0].set_ylabel("units")

by_month = features.groupby("month")["y"].mean()
axes[1].plot(by_month.index, by_month.to_numpy(), marker="o", color="#55A868")
axes[1].set_xticks(range(1, 13))
axes[1].set_title("Mean sales by month"); axes[1].set_xlabel("month")
fig.suptitle("Seasonality", fontweight="bold")
fig.tight_layout()
save(fig, "03_favorita_seasonality");

## Promotion effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
promo = features["onpromotion"] > 0
axes[0].boxplot([features.loc[~promo, "y"], features.loc[promo, "y"]],
                labels=["no promo", "on promo"], showfliers=False)
axes[0].set_title("Sales with and without promotion"); axes[0].set_ylabel("units")

bins = pd.cut(features["onpromotion"], bins=[-0.1, 0, 1, 5, 20, 1e9],
              labels=["0", "1", "2-5", "6-20", "20+"])
# observed=False keeps empty bins, so the labels and heights stay the same length.
by_promo = features.groupby(bins, observed=False)["y"].mean().dropna()
axes[1].bar(by_promo.index.astype(str), by_promo.to_numpy(),
            color="#DD8452", edgecolor="white")
axes[1].set_title("Mean sales by promotion intensity")
axes[1].set_xlabel("items on promotion")

lift = features.loc[promo, "y"].mean() / max(features.loc[~promo, "y"].mean(), 1e-9)
fig.suptitle(f"Promotion effect (mean lift {lift:.2f}x)", fontweight="bold")
fig.tight_layout()
save(fig, "03_favorita_promotion_effect");

## Rolling-origin fold diagram

Folds walk forward in time. A random split would train on the future to predict
the past and report an accuracy the model can never achieve live.


In [ ]:
folds = make_rolling_origin_splits(features, n_splits=cfg.n_splits, horizon=cfg.horizon)

fig, ax = plt.subplots(figsize=(13, 3.2))
for i, (train_idx, val_idx) in enumerate(folds):
    tr, va = features["ds"].iloc[train_idx], features["ds"].iloc[val_idx]
    ax.barh(i, (tr.max() - tr.min()).days, left=tr.min(), height=0.55,
            color="#4C72B0", label="train" if i == 0 else None)
    ax.barh(i, (va.max() - va.min()).days + 1, left=va.min(), height=0.55,
            color="#C44E52", label="validation" if i == 0 else None)
    assert tr.max() < va.min()
ax.set_yticks(range(len(folds)))
ax.set_yticklabels([f"fold {i}" for i in range(len(folds))])
ax.set_title(f"Rolling-origin splits: {cfg.n_splits} folds, horizon {cfg.horizon} days",
             fontweight="bold")
ax.set_xlabel("date"); ax.legend(loc="lower left")
fig.tight_layout()
save(fig, "03_favorita_rolling_origin_folds");

pd.DataFrame([
    {"fold": i,
     "train_rows": len(t), "train_end": features["ds"].iloc[t].max().date(),
     "val_rows": len(v), "val_start": features["ds"].iloc[v].min().date(),
     "val_end": features["ds"].iloc[v].max().date()}
    for i, (t, v) in enumerate(folds)
])

## Oil, and the feature inventory

In [ ]:
oil = prepare_oil(frames["oil"], features["ds"])
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(oil["date"], oil["oil_price"], color="#937860", lw=1)
ax.set_title("WTI oil price, forward-filled across weekend gaps", fontweight="bold")
ax.set_xlabel("date"); ax.set_ylabel("USD/bbl")
fig.tight_layout()
save(fig, "03_favorita_oil_price");

groups = {
    "identifiers": ["unique_id", "ds", "y"],
    "calendar": [c for c in features if c in
                 {"dow","dom","month","week_of_year","year","is_weekend","is_payday"}],
    "fourier": [c for c in features if c.startswith("fourier_")],
    "holiday": [c for c in features if "holiday" in c or c == "is_work_day"],
    "lags/rolling": [c for c in features if c.startswith(("lag_", "roll_"))],
    "exogenous": [c for c in features if c in
                  {"onpromotion","oil_price","city","state","type","cluster",
                   "store_nbr","family"}],
}
for name, columns in groups.items():
    print(f"{name:<14} {len(columns):>2}  {', '.join(columns[:6])}"
          f"{' ...' if len(columns) > 6 else ''}")
print(f"\nTOTAL {features.shape[1]} columns, {len(features):,} rows")